# NB23 · Standard-split confusion matrices for the two production models

Purpose: fill the two empty cells of Table 5.6 (false-positive and true-positive rate of the hybrid model) and add precision for both models, so that every metric in Table 5.6 matches the metrics set in Chapter 3.

No training. The notebook reloads the final checkpoints written by NB10 (hybrid) and NB14 (neural), scores the 1,093 articles of the test split, and checks that Macro-F1, accuracy, AUC and the error count reproduce the published values before any new number is reported.

Inputs (Add input): the NB10 notebook output (ckpt/last.pt and chunks_K9.npz), the NB14 notebook output (ckpt_neural/last.pt), dataset.parquet and vstat16_scaled.parquet. GPU required.

In [ ]:
# ============================ 1/6  configuration ============================
import os, glob, time, numpy as np, pandas as pd, torch

P_DATASET = "/kaggle/input/datasets/bahaaqassem/aig-and-humang-dataset/dataset.parquet"
P_VSTAT16 = "/kaggle/input/datasets/bahaaqassem/aig-16-features/vstat16_scaled.parquet"
P_CHUNKS  = "/kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/chunks_K9.npz"
CKPTS = {
    "hybrid": "/kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/ckpt/last.pt",
    "neural": "/kaggle/input/notebooks/bahaaqassem/nb14-neural-only-production/ckpt_neural/last.pt",
}

# values already published in Table 5.6; the run must reproduce them before anything new is read
EXPECTED = {
    "hybrid": {"macro_f1": 99.82, "acc": 99.82, "auc": 100.00, "errors": 2},
    "neural": {"macro_f1": 99.91, "acc": 99.91, "auc": 100.00, "errors": 1,
               "TN": 538, "FP": 1, "FN": 0, "TP": 554},
}

MODEL_ID  = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
STAT_COLS = ["burstiness", "ttr", "quote_ratio", "function_word_ratio", "compressibility"]
K_CHUNKS, MAX_CT, STRIDE = 9, 510, 460
HIDDEN, DROPOUT = 256, 0.0
OUT_PRED = "/kaggle/working/nb23_test_predictions.parquet"
OUT_MET  = "/kaggle/working/nb23_test_metrics.parquet"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[1/6] device {DEV}", flush=True)

def resolve(path):
    """Use the declared path; if the input was mounted under another name, find the file by its name."""
    if os.path.exists(path):
        return path
    name = os.path.basename(path)
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    parent = os.path.basename(os.path.dirname(path))
    hits = [h for h in hits if os.path.basename(os.path.dirname(h)) == parent] or hits
    return hits[0] if hits else None

P_DATASET, P_VSTAT16, P_CHUNKS = resolve(P_DATASET), resolve(P_VSTAT16), resolve(P_CHUNKS)
CKPTS = {k: resolve(v) for k, v in CKPTS.items()}
for name, p in [("dataset", P_DATASET), ("vstat16", P_VSTAT16), ("chunks", P_CHUNKS)] + list(CKPTS.items()):
    print(f"[1/6]   {name:8s} {'found  ' if p else 'MISSING'} {p}", flush=True)
assert all([P_DATASET, P_VSTAT16, P_CHUNKS] + list(CKPTS.values())), "add the missing inputs first"
print("[1/6] configuration ready", flush=True)


In [ ]:
# ============================ 2/6  data, aligned exactly as in NB10 ============================
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
v16 = pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16 = v16.set_index("article_id")
df = df.loc[df.index.intersection(v16.index)]; v16 = v16.loc[df.index]
assert (df.index == v16.index).all()
print(f"[2/6] aligned articles {len(df)}", flush=True)

Xstat = v16[STAT_COLS].to_numpy(np.float32)
y     = df["label"].to_numpy(np.int64)
gen   = df["generator"].fillna("__human__").to_numpy()
split = df["split"].to_numpy()
texts = df["text"].astype(str).tolist()

z = np.load(P_CHUNKS, allow_pickle=True); CH = z["ch"]; NCH = z["nch"]
print(f"[2/6] chunk cache {CH.shape} | n_chunks {NCH.shape}", flush=True)
assert CH.shape[0] == len(df) == NCH.shape[0], "chunk cache does not match the aligned dataset"

te = np.where(split == "test")[0]
print(f"[2/6] test split {len(te)} | human {int((y[te]==0).sum())} | machine {int((y[te]==1).sum())}", flush=True)
print(f"[2/6] test machine by generator: {pd.Series(gen[te][y[te]==1]).value_counts().to_dict()}", flush=True)
assert len(te) == 1093, "the test split must contain 1,093 articles"


In [ ]:
# ============================ 3/6  check that the cached chunks belong to these articles ============================
!pip install -q transformers
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
CHUNK_LEN = MAX_CT + 2
print(f"[3/6] tokenizer loaded | PAD {PAD}", flush=True)

def to_chunks(text):   # NB10 cell 6, verbatim
    ids = tok(text, add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
    wins = [ids[i:i+MAX_CT] for i in range(0, len(ids), STRIDE)][:K_CHUNKS] or [ids[:MAX_CT]]
    out = []
    for w in wins:
        seq = [CLS] + w + [SEP]
        seq = seq + [PAD]*(CHUNK_LEN-len(seq))
        out.append(seq)
    return out

rng = np.random.default_rng(42)
probe = rng.choice(te, size=20, replace=False)
bad = 0
for i in probe:
    ch = to_chunks(texts[i])
    same = (len(ch) == int(NCH[i])) and all((CH[i, j] == np.array(c)).all() for j, c in enumerate(ch))
    bad += (not same)
print(f"[3/6] re-tokenized {len(probe)} random test articles: {len(probe)-bad} match the cache, {bad} differ", flush=True)
assert bad == 0, "cache rows do not match the articles in this order; stop"


In [ ]:
# ============================ 4/6  the model, as loaded in NB21 ============================
import torch.nn as nn
from transformers import AutoModel

class Net(nn.Module):
    def __init__(self, use_stat, stat_dim=5):
        super().__init__()
        self.use_stat = use_stat
        self.enc = AutoModel.from_pretrained(MODEL_ID)
        h = self.enc.config.hidden_size; fused = h + (stat_dim if use_stat else 0)
        self.register_buffer("mu", torch.zeros(fused)); self.register_buffer("sd", torch.ones(fused))
        self.head = nn.Sequential(nn.Linear(fused, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(),
                                  nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 2))
    def encode(self, ids, nch):
        B, K, L = ids.shape; flat = ids.view(B*K, L); att = (flat != PAD).long()
        cls = self.enc(input_ids=flat, attention_mask=att).last_hidden_state[:, 0, :].view(B, K, -1)
        m = (torch.arange(K, device=ids.device)[None, :] < nch[:, None]).float().unsqueeze(-1)
        return (cls*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self, ids, nch, stat=None):
        v = self.encode(ids, nch)
        if self.use_stat: v = torch.cat([v, stat], 1)
        v = (v - self.mu)/self.sd
        return self.head(v)

@torch.no_grad()
def predict(model, rows, tag, bs=8):
    model.eval(); P = []; t0 = time.time()
    for b in range(0, len(rows), bs):
        r = rows[b:b+bs]
        ids = torch.from_numpy(CH[r].astype(np.int64)).to(DEV)
        nch = torch.tensor(NCH[r].astype(np.int64)).to(DEV)
        st  = torch.from_numpy(Xstat[r]).to(DEV)
        with torch.amp.autocast("cuda", enabled=(DEV == "cuda")):
            p = torch.softmax(model(ids, nch, st), 1)[:, 1]
        P.append(p.float().cpu().numpy())
        if (b//bs) % 25 == 0:
            print(f"[5/6]   {tag} {b+len(r)}/{len(rows)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(P)
print("[4/6] model class and prediction routine ready", flush=True)


In [ ]:
# ============================ 5/6  score the test split with both checkpoints ============================
probs = {}
for name, ck in CKPTS.items():
    sd = torch.load(ck, map_location=DEV, weights_only=False)
    state = sd["model"] if isinstance(sd, dict) and "model" in sd else sd
    m = Net(name == "hybrid").to(DEV)
    m.load_state_dict(state)
    fused = m.mu.numel()
    print(f"[5/6] {name}: checkpoint loaded | fused dimension {fused}", flush=True)
    assert fused == (773 if name == "hybrid" else 768), "checkpoint does not match the declared model type"
    probs[name] = predict(m, te, name)
    print(f"[5/6] {name}: scored {len(probs[name])} test articles", flush=True)
    del m; torch.cuda.empty_cache()

PRED = pd.DataFrame({"article_id": df.index[te], "label": y[te], "generator": gen[te],
                     "p_hybrid": probs["hybrid"], "p_neural": probs["neural"]})
PRED["pred_hybrid"] = (PRED.p_hybrid >= 0.5).astype(int)
PRED["pred_neural"] = (PRED.p_neural >= 0.5).astype(int)
PRED.to_parquet(OUT_PRED, index=False)
print(f"[5/6] wrote {OUT_PRED} | rows {len(PRED)}", flush=True)


In [ ]:
# ============================ 6/6  confusion matrices, Table 5.6 metrics, reproduction check ============================
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score, confusion_matrix

rows = []
for name in ("hybrid", "neural"):
    yt, pr, p = PRED.label.to_numpy(), PRED[f"pred_{name}"].to_numpy(), PRED[f"p_{name}"].to_numpy()
    TN, FP, FN, TP = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
    r = {"model": name,
         "TN": int(TN), "FP": int(FP), "FN": int(FN), "TP": int(TP), "errors": int(FP + FN),
         "macro_f1":        round(100*f1_score(yt, pr, average="macro"), 2),
         "acc":             round(100*accuracy_score(yt, pr), 2),
         "auc":             round(100*roc_auc_score(yt, p), 2),
         "FPR":             round(100*FP/(FP+TN), 2),
         "TPR":             round(100*TP/(TP+FN), 2),
         "precision_machine": round(100*precision_score(yt, pr, pos_label=1, zero_division=0), 2),
         "recall_machine":    round(100*recall_score(yt, pr, pos_label=1, zero_division=0), 2),
         "precision_human":   round(100*precision_score(yt, pr, pos_label=0, zero_division=0), 2),
         "recall_human":      round(100*recall_score(yt, pr, pos_label=0, zero_division=0), 2),
         "macro_precision": round(100*precision_score(yt, pr, average="macro", zero_division=0), 2),
         "macro_recall":    round(100*recall_score(yt, pr, average="macro", zero_division=0), 2)}
    rows.append(r)
    print(f"[6/6] {name}: TN {r['TN']} FP {r['FP']} FN {r['FN']} TP {r['TP']} | errors {r['errors']}", flush=True)

M = pd.DataFrame(rows).set_index("model")
M.reset_index().to_parquet(OUT_MET, index=False)
print(f"[6/6] wrote {OUT_MET}", flush=True)

print("\n[6/6] REPRODUCTION CHECK against the published Table 5.6", flush=True)
ok = True
for name, exp in EXPECTED.items():
    for k, v in exp.items():
        got = M.loc[name, k]
        hit = abs(float(got) - float(v)) < 0.006
        ok &= hit
        print(f"[6/6]   {name:6s} {k:9s} published {v:>7} | now {got:>7} | {'match' if hit else 'DIFFERENT'}", flush=True)
print(f"[6/6] reproduction {'PASSED - the new numbers below can be reported' if ok else 'FAILED - do not report until resolved'}", flush=True)

print("\n[6/6] TABLE 5.6, complete (per cent unless stated)", flush=True)
T = pd.DataFrame({
    "Metric": ["Macro-F1", "Accuracy", "AUC", "Errors out of 1,093 articles",
               "False-positive rate", "True-positive rate (recall, machine class)", "Precision (machine class)"],
    "Hybrid (773)":      [f"{M.loc['hybrid', c]:.0f}" if c == "errors" else f"{M.loc['hybrid', c]:.2f}"
                          for c in ("macro_f1", "acc", "auc", "errors", "FPR", "TPR", "precision_machine")],
    "Neural only (768)": [f"{M.loc['neural', c]:.0f}" if c == "errors" else f"{M.loc['neural', c]:.2f}"
                          for c in ("macro_f1", "acc", "auc", "errors", "FPR", "TPR", "precision_machine")],
})
print(T.to_string(index=False), flush=True)

print("\n[6/6] all metrics", flush=True)
print(M.to_string(), flush=True)

print("\n[6/6] misclassified articles", flush=True)
for name in ("hybrid", "neural"):
    E = PRED[PRED[f"pred_{name}"] != PRED.label]
    for _, e in E.iterrows():
        kind = "false alarm (human called machine)" if e.label == 0 else "missed machine article"
        print(f"[6/6]   {name:6s} {e.article_id:24s} generator {e.generator:10s} p={e[f'p_{name}']:.4f}  {kind}", flush=True)
print("[6/6] done", flush=True)
